In [ ]:
# Walk through each directory in the folder tree
for folder_name, _, filenames in os.walk(top_folder):

    # Check if .gz file 'lissajous_data.gz not in the directory
    if 'lissajous_data.gz' not in filenames:
        print(f"Folder {folder_name} does not contain a .gz file")
        continue
    
    else:
        print(f"Processing folder: {folder_name}")

        # Delete the .png and .csv files that are in the folder
        for file in os.listdir(folder_name):
            if file.endswith(".png") or file.endswith(".csv"):
                os.remove(os.path.join(folder_name, file))
        
        # Create result .csv file
        result_csv_path = os.path.join(
            folder_name,
            'lissajous_calculations.csv'
        )

        with open(
            result_csv_path,
            mode='w',
            newline="",
            encoding='utf-8'
        ) as csvfile:
            csv_writer = csv.writer(csvfile)
            csv_writer.writerow(column_names)

        # Read the .gz file
        data_df = pd.read_csv(
            os.path.join(folder_name, 'lissajous_data.csv.gz'),
            compression='gzip',
            encoding='uft-8'
        )

        # Loop over the grouped data by the column 'psdata_file_name'
        for file, data in data_df.groupby('psdata_file_name'):

                # Smoothing voltage and current curve
                smoothvoltage = savgol_filter(data[:, U], 5001, 3)
                smoothcurrent1 = savgol_filter(data[:, Ip], 1001, 3)

                # Look in column charge (Q) for all the sign changes
                # and save as 'zero_Q' (Q=0)
                smoothcharge = savgol_filter(data[:, Q], 5001, 3)
                zero_Q = np.where(np.diff(np.sign(smoothcharge)) != 0)

                # Then look in column voltage (U) for the data that
                # are in the same row (Umin)
                zero_dataUmin = data[zero_Q, U]

                # Find the rows that contain pos and neg Umin values
                # in the whole Umin data set.
                posUminrows = np.where(zero_dataUmin > 0)
                negUminrows = np.where(zero_dataUmin < 0)

                # Make arrays containing the Umin data points
                # from the corresponding row numbers.
                posUmindata = zero_dataUmin[posUminrows]
                negUmindata = zero_dataUmin[negUminrows]

                # Calculate the average for both pos and neg Umin.
                posUmin = np.mean(posUmindata)
                negUmin = np.mean(negUmindata)

                # Calculating slopes
                #    B---------------A
                #   /               /
                #  /               /
                # C---------------D

                # Calculating one period (correction to s)
                Period = 1 / f

                # Calculating row of first period start
                t0 = np.where(np.diff(np.sign(data[:, t])) > 0)[0][0]

                # Calculating row of period endings
                t1 = np.argmin(np.abs(data[:, t] - Period * 1))
                t2 = np.argmin(np.abs(data[:, t] - Period * 2))
                t3 = np.argmin(np.abs(data[:, t] - Period * 3))
                t4 = np.argmin(np.abs(data[:, t] - Period * 4))

                # Calculation rows per period
                Periodrows = int(
                    np.floor(
                        np.mean(
                            [t1, t2, t3, t4]
                            - np.array([t0, t1, t2, t3])
                        )
                    )
                )

                # Calculating Voltage values and row numbers of
                # points B and D
                U_A = np.max(data[t0:t1, U])

                U_C = np.min(data[t0:t1, U])

                AU = t0 + int(
                    np.floor(
                        np.mean(
                            np.where(
                                data[t0:t1, U] == U_A
                            )
                        )
                    )
                )

                CU = t0 + int(
                    np.floor(
                        np.mean(
                            np.where(
                                data[t0:t1, U] == U_C
                            )
                        )
                    )
                )

                U_A1 = np.max(data[t1:t2, U])

                AU1 = t1 + int(
                    np.floor(
                        np.mean(
                            np.where(
                                data[t1:t2, U] == U_A1
                            )
                        )
                    )
                )

                # Calculate tops of charge curves Lissajous
                Q_A = np.max(data[t0:t1, Q])

                Q_C = np.min(data[t0:t1, Q])

                AQ = t0 + int(
                    np.floor(
                        np.mean(
                            np.where(
                                data[t0:t1, Q] == Q_A
                            )
                        )
                    )
                )

                CQ = t0 + int(
                    np.floor(
                        np.mean(
                            np.where(
                                data[t0:t1, Q] == Q_C
                            )
                        )
                    )
                )

                # calculate tops of Lissajous
                linfit_AB = []

                linfit_BC = []

                dx = int(np.floor(1 / 16 * Periodrows))

                dy = int(np.floor(1 / 32 * Periodrows))

                step = 100

                for jj in range(AQ + dx, CU - dx, step):

                    fitobject_AB = np.polyfit(
                        smoothvoltage[AQ:jj],
                        smoothcharge[AQ:jj],
                        1
                    )

                    R2_AB = np.corrcoef(
                        smoothvoltage[AQ:jj],
                        smoothcharge[AQ:jj]
                    )[0, 1] ** 2

                    fitobject_BC = np.polyfit(
                        smoothvoltage[jj:CU],
                        smoothcharge[jj:CU],
                        1
                    )

                    R2_BC = np.corrcoef(
                        smoothvoltage[jj:CU],
                        smoothcharge[jj:CU]
                    )[0, 1] ** 2

                    linfit_AB.append(R2_AB)

                    linfit_BC.append(R2_BC)

                linfit = []

                for kk in range(len(linfit_AB)):

                    linfit.append(
                        (1 - linfit_AB[kk])
                        + (1 - linfit_BC[kk])
                    )

                val, ind = np.min(linfit), np.argmin(linfit)
                B = ind * step + AQ + dx

                B1 = B + 1 * Periodrows
                B2 = B + 2 * Periodrows
                B3 = B + 3 * Periodrows

                linfit_CD = []
                linfit_DA = []

                for jj in range(CQ + dx, AU1 - dx, step):

                    fitobject_CD = np.polyfit(
                        smoothvoltage[CQ:jj],
                        smoothcharge[CQ:jj],
                        1
                    )

                    R2_CD = np.corrcoef(
                        smoothvoltage[CQ:jj],
                        smoothcharge[CQ:jj]
                    )[0, 1] ** 2

                    fitobject_DA = np.polyfit(
                        smoothvoltage[jj:AU1],
                        smoothcharge[jj:AU1],
                        1
                    )

                    R2_DA = np.corrcoef(
                        smoothvoltage[jj:AU1],
                        smoothcharge[jj:AU1]
                    )[0, 1] ** 2

                    linfit_CD.append(R2_CD)

                    linfit_DA.append(R2_DA)

                linfit = []

                for kk in range(len(linfit_CD)):

                    linfit.append(
                        (1 - linfit_CD[kk])
                        + (1 - linfit_DA[kk])
                    )

                val, ind = np.min(linfit), np.argmin(linfit)

                D = ind * step + CQ + dx

                D1 = D + 1 * Periodrows
                D2 = D + 2 * Periodrows
                D3 = D + 3 * Periodrows

                AQ1 = AQ + 1 * Periodrows
                AQ2 = AQ + 2 * Periodrows
                AQ3 = AQ + 3 * Periodrows
                AQ4 = AQ + 4 * Periodrows
                AU4 = AU + 4 * Periodrows
                AU1 = AU + 1 * Periodrows
                AU2 = AU + 2 * Periodrows
                AU3 = AU + 3 * Periodrows
                AU4 = AU + 4 * Periodrows

                CQ1 = CQ + 1 * Periodrows
                CQ2 = CQ + 2 * Periodrows
                CQ3 = CQ + 3 * Periodrows
                CU1 = CU + 1 * Periodrows
                CU2 = CU + 2 * Periodrows
                CU3 = CU + 3 * Periodrows

                # Finding the values of the first order regression
                # for the data points
                AB = np.polyfit(smoothvoltage[AQ:B], smoothcharge[AQ:B], 1)
                BC = np.polyfit(smoothvoltage[B:CU], smoothcharge[B:CU], 1)
                CD = np.polyfit(smoothvoltage[CQ:D], smoothcharge[CQ:D], 1)
                DA1 = np.polyfit(smoothvoltage[D:AU1], smoothcharge[D:AU1], 1)
                A1B1 = np.polyfit(smoothvoltage[AQ1:B1], smoothcharge[AQ1:B1], 1)
                B1C1 = np.polyfit(smoothvoltage[B1:CU1], smoothcharge[B1:CU1], 1)
                C1D1 = np.polyfit(smoothvoltage[CQ1:D1], smoothcharge[CQ1:D1], 1)
                D1A2 = np.polyfit(smoothvoltage[D1:AU2], smoothcharge[D1:AU2], 1)
                A2B2 = np.polyfit(smoothvoltage[AQ2:B2], smoothcharge[AQ2:B2], 1)
                B2C2 = np.polyfit(smoothvoltage[B2:CU2], smoothcharge[B2:CU2], 1)
                C2D2 = np.polyfit(smoothvoltage[CQ2:D2], smoothcharge[CQ2:D2], 1)
                D2A3 = np.polyfit(smoothvoltage[D2:AU3], smoothcharge[D2:AU3], 1)
                A3B3 = np.polyfit(smoothvoltage[AQ3:B3], smoothcharge[AQ3:B3], 1)
                B3C3 = np.polyfit(smoothvoltage[B3:CU3], smoothcharge[B3:CU3], 1)
                C3D3 = np.polyfit(smoothvoltage[CQ3:D3], smoothcharge[CQ3:D3], 1)
                D3A4 = np.polyfit(smoothvoltage[D3:AU4], smoothcharge[D3:AU4], 1)

                # Calculating the average of the 4 periods.
                meanAB = np.mean([AB, A1B1, A2B2, A3B3], axis=0)
                meanBC = np.mean([BC, B1C1, B2C2, B3C3], axis=0)
                meanCD = np.mean([CD, C1D1, C2D2, C3D3], axis=0)
                meanDA = np.mean([DA1, D1A2, D2A3, D3A4], axis=0)

                # Making arrays to plot the fitted lines.
                xline = np.arange(-10e3, 10.1e3, 1e2)
                lineAB = xline * meanAB[0] + meanAB[1]
                lineBC = xline * meanBC[0] + meanBC[1]
                lineCD = xline * meanCD[0] + meanCD[1]
                lineDA = xline * meanDA[0] + meanDA[1]

                # Plotting the data and fitted lines
                plt.plot(data[t0:t4, U], data[t0:t4, Q], label="Data")
                plt.plot(xline, lineAB, label="AB")
                plt.plot(xline, lineBC, label="BC")
                plt.plot(xline, lineCD, label="CD")
                plt.plot(xline, lineDA, label="DA")
                plt.axis([-15e3, 15e3, -1e-6, 1e-6])
                plt.xlabel("Voltage (V)")
                plt.ylabel("Charge (µC)")
                plt.legend()

                # Save the plot to a .png file in the result folder
                plot_filename = (os.path.splitext(file)[0]
                                 + "_generated_plot.png")
                plt.savefig(os.path.join(folder_name, plot_filename))

                # clear the current plot to avoid plotting data
                # of the previous iteration
                plt.clf()

                # Detect discharges
                smoothcurrent1 = savgol_filter(data[:, Ip], 10001, 3)
                correctedIp = data[:, Ip] - smoothcurrent1
                smoothcurrent2 = savgol_filter(correctedIp, 31, 3)
                smoothcurrent2[smoothcurrent2 < discharge_treshold] = 0

                # Calculating first derivative of current
                # to detect pulses in DA region
                eafgI1 = np.zeros(30)
                for jj in range(D, AQ1):
                    eafgI1 = np.append(
                        eafgI1,
                        (np.mean(smoothcurrent2[jj:(jj+30)])
                         - np.mean(smoothcurrent2[(jj-30):jj]))
                         / (30 * (data[1, t] - data[0, t]))
                    )

                eafgI2 = np.zeros(30)
                for jj in range(D1, AQ2):
                    eafgI2 = np.append(
                        eafgI2,
                        (np.mean(smoothcurrent2[jj:(jj+30)])
                         - np.mean(smoothcurrent2[(jj-30):jj]))
                         / (30 * (data[1, t] - data[0, t]))
                )

                eafgI3 = np.zeros(30)
                for jj in range(D2, AQ3):
                    eafgI3 = np.append(
                        eafgI3,
                        (np.mean(smoothcurrent2[jj:(jj+30)])
                         - np.mean(smoothcurrent2[(jj-30):jj]))
                         / (30 * (data[1, t] - data[0, t]))
                    )

                eafgI4 = np.zeros(30)
                for jj in range(D3, AQ4):
                    eafgI4 = np.append(
                        eafgI4,
                        (np.mean(smoothcurrent2[jj:(jj+30)])
                         - np.mean(smoothcurrent2[(jj-30):jj]))
                         / (30 * (data[1, t] - data[0, t]))
                    )

                # Find actual sign changes
                Num_zerosI1 = np.diff(np.sign(eafgI1))
                indx_downI1 = np.where(Num_zerosI1 < 0)[0]
                Num_zerosI2 = np.diff(np.sign(eafgI2))
                indx_downI2 = np.where(Num_zerosI2 < 0)[0]
                Num_zerosI3 = np.diff(np.sign(eafgI3))
                indx_downI3 = np.where(Num_zerosI3 < 0)[0]
                Num_zerosI4 = np.diff(np.sign(eafgI4))
                indx_downI4 = np.where(Num_zerosI4 < 0)[0]

                # Counting times that first derivative
                # has zero points from pos to neg
                Num_peaksI = (
                    len(indx_downI1)
                    + len(indx_downI2)
                    + len(indx_downI3)
                    + len(indx_downI4)
                )
                Av_num_peaksI = round(Num_peaksI / 4)

                # Displaced charge per cycle (C)
                Q_disp = np.mean(lineAB - lineCD)

                # Displaced charge per peak (C/peak)
                Av_Q_peak = Q_disp / Av_num_peaksI

                # Calculating plasma power (W)
                plasmapower = np.mean(data[t0:t4, U] * data[t0:t4, Ip])

                # Calculating power source power (W)
                sourcepower = np.mean(data[t0:t4, U] * data[t0:t4, Ib])

                # Calculating RMS current plasma (A)
                RMSIp = np.sqrt(np.mean(np.square(data[t0:t4, Ip])))

                # Calculating RMS current source (A)
                RMSIb = np.sqrt(np.mean(np.square(data[t0:t4, Ib])))

                # Calculating Upp (V)
                Upp = abs(U_A) + abs(U_C)

                # Calculate burning voltage (V)
                U_burning_pos = (
                    (1 - meanCD[0] / C_diel)
                    / (1 - meanCD[0] / meanDA[0])
                    * (posUmin - negUmin)
                    / 2
                )
                
                U_burning_neg = (
                    (1 - meanAB[0] / C_diel)
                    / (1 - meanAB[0] / meanBC[0])
                    * (posUmin - negUmin)
                    / 2
                )
                
                # Calculate breakdown voltage negative
                U_breakdown_neg = 1 / (1 + C_gas/C_diel) * negUmin

                # Calculate breakdown voltage positive
                U_breakdown_pos = 1 / (1 + C_gas/C_diel) * posUmin

                with open(
                    result_csv_path,
                    mode="a",
                    newline="",
                    encoding='utf-8'
                ) as csvfile:
                    csv_writer = csv.writer(csvfile)
                    csv_writer.writerow(
                        [
                        file,
                        data['project_name'],
                        data['reaction_type'],
                        data['material_supplier'],
                        data['material_name'],
                        data['wattage_const'],
                        data['residence_time_s'],
                        data['measurement_number'],
                        data['plasma_state'],
                        data['date'],
                        posUmin,
                        negUmin,
                        Upp,
                        meanAB[0],
                        meanBC[0],
                        meanCD[0],
                        meanDA[0],
                        plasmapower,
                        sourcepower,
                        RMSIp,
                        RMSIb,
                        Av_num_peaksI,
                        Av_Q_peak,
                        Q_disp,
                        U_burning_neg,
                        U_burning_pos,
                        U_breakdown_neg,
                        U_breakdown_pos
                        ]
                    )

        # Read the result .csv file for outlier detection
        data = pd.read_csv(result_csv_path, encoding='utf-8')
        
        try:
            # Fit an EllipticEnvelope to the data
            clf = EllipticEnvelope(contamination=0.15, random_state=42)

            columns_to_predict_outliers = [
                'U_min_pos_V',
                'U_min_neg_V',
                'U_pp_V',
                'slope_AB_F',
                'slope_BC_F',
                'slope_CD_F',
                'slope_DA_F',
                'plasma_power_W',
                'plasma_rms_current_A',
                'displaced_charge_C',
                'U_burning_neg_V',
                'U_burning_pos_V',
                'U_breakdown_neg_V',
                'U_breakdown_pos_V'
            ]

            outlier_predictions = clf.fit_predict(
                data[columns_to_predict_outliers]
            )

            # Add a new column 'outlier' to the data
            # with the outlier predictions
            data['outlier'] = outlier_predictions

            # Filter the outliers
            no_outliers = data[data['outlier'] == 1].drop(columns=['outlier'])

            # Create result without overliers csv file
            no_outlier_csv_path = os.path.join(
                folder_name,
                'lissajous_calculations_no_outliers.csv'
            )

            # Write the no_outliers DataFrame to a CSV file
            no_outliers.to_csv(
                no_outlier_csv_path,
                index=False,
                encoding='utf-8'
            )

        except Exception as e:
            print("An error occurred while processing for outlier detection:"
                  f"{result_csv_path}. Error message: {str(e)}")
        
        # Lissajous figure generation and smoothing
        # Saving the data to a .csv file
        # Start from the data_df DataFrame
        
        
        
        # Get a list of all .csv files in the folder
        csv_files = [
            os.path.splitext(file)[0]
            + ".csv" for file in psdata_files
        ]

        try:
            # Initialize lists to store DataFrames
            data_frames = []
            data_info_frames = []

            # Read each file and store them in the lists
            for file in csv_files:
                # Extract information from the file path and name
                info = parse_file_path(file)

                # Convert the information dictionary to a DataFrame
                info_df = pd.DataFrame([info])

                # Read the CSV file
                data_temp = pd.read_csv(file)

                # Select every 125th row and store it in a new DataFrame
                data_temp_subset = data_temp.iloc[::125, :]

                # Match the length of info_df to data_temp_subset
                repeated_info_df = pd.concat(
                    [info_df] * len(data_temp_subset),
                    ignore_index=True
                )

                # Concatenate the information DataFrame with the data subset DataFrame
                data_info_temp = pd.concat(
                    [
                        repeated_info_df,
                        data_temp_subset.reset_index(drop=True)
                    ],
                    axis=1
                )

                # Append to the data_info list
                data_info_frames.append(data_info_temp)

                # Append the original data to the data list
                data_frames.append(data_temp)

            # Concatenate all DataFrames in the lists
            data_info = pd.concat(data_info_frames, ignore_index=True)
            data = pd.concat(data_frames, ignore_index=True)

            # Write the subset data to a .csv file
            output_csv_path = os.path.join(
                folder_name,
                "lissajous_data_subset.csv"
            )
            data_info.to_csv(output_csv_path, index=False)

            # Calculate the mean of all the measurements grouped by 'time_s'
            data_mean = data.groupby('time_s').mean().reset_index()

            # Select at random 6250 of the rows of the average DataFrame
            # as a simple smoothing technique
            data_new = (
                data_mean
                .sample(n=6250)
                .sort_values(by="time_s")
                .reset_index(drop=True)
            )

            # Calculate a rolling gaussian mean
            # of the 'voltage_V' and 'charge_C' columns
            window_size = 63

            data_new['voltage_V_smooth'] = (
                data_new['voltage_V']
                .rolling(window_size, win_type='gaussian', center=True)
                .mean(std=window_size/2)
            )

            data_new['charge_C_smooth'] = (
                data_new['charge_C']
                .rolling(window_size, win_type='gaussian', center=True)
                .mean(std=window_size/2)
            )

            # Calculate the derivative of the 'charge_C_smooth' column
            # to the'voltage_V_smooth' column
            data_new['dQ_dV'] = np.abs(
                np.gradient(
                    data_new['charge_C_smooth'],
                    data_new['voltage_V_smooth']
                )
            )

            # Drop the unwanted columns
            data_new.drop(
                columns=[
                    "current_source_A",
                    "power_plasma_W",
                    "power_source_W"
                    ],
                inplace=True
            )

            # Repeat the info_df to match the length of data_new
            info_df = pd.concat([info_df]*len(data_new), ignore_index=True)

            # Concatenate the information DataFrame
            # with the data DataFrame
            data_new = pd.concat([info_df, data_new], axis=1)

            # Write data_new to a .csv file in the folder
            # where the .csv files for calculation come from
            output_csv_path = os.path.join(
                folder_name,
                "smoothed_lissajous_and_derivative.csv"
            )

            data_new.to_csv(output_csv_path, index=False)

            # Create a plot of the Lissajous figure
            # and the derivative of the smoothed charge to the voltage
            plt.figure(figsize=(10, 5))
            
            # Plot the Lissajous figure and save it to a .png file
            plt.subplot(1, 2, 1)
            plt.plot(
                data_new["voltage_V"],
                data_new["charge_C"],
                label="Lissajous"
            )
            plt.xlabel("Voltage (V)")
            plt.ylabel("Charge (C)")
            plt.title("Lissajous Figure: Charge vs Voltage")
            plt.axis([-15e3, 15e3, -1e-6, 1e-6])
            plt.legend()

            # Plot the derivative of the charge to the voltage
            # and save it to a .png file
            plt.subplot(1, 2, 2)
            plt.plot(data_new['time_s'], data_new['dQ_dV'], label="dQ/dV")
            plt.xlabel("Time (s)")
            plt.ylabel("dQ/dV (C/V)")
            plt.title("Derivative of Charge to Voltage")
            plt.axis([0, 1e-3, 0, 300e-12])

            # Save the plot to a .png file in the folder
            # where the .csv files for calculation come from
            output_png_path = os.path.join(
                folder_name,
                "smoothed_lissajous_and_derivative_plot.png"
            )
            plt.savefig(output_png_path)
            plt.close()  # Close the plot to free up resources

        except Exception as e:
            print("An error occurred while averaging the lissajous figures."
                  f"Error message: {str(e)}")

        # Current profile generation
        # We will select every nth line of the current profile
        # to save data space and it works as a simple smoothing
        n = 63

        # Negative discharges
        try:
            all_results_neg = []

            for index, file in enumerate(data_frames):
                # Get information from the file path of the CSV file
                info = parse_file_path(psdata_files[index])

                # Convert the information dictionary to a DataFrame
                info_df = pd.DataFrame([info])

                file_results_neg = []
                
                for i in range(0,6):
                    data_voltage_current_neg = file[
                        (file["time_s"] >= 0.115e-3 + i * (1/f))
                        & (file["time_s"] < 0.115e-3 + (i+0.5) * (1/f))
                        ]
                    
                    data_voltage_current_neg = (
                        data_voltage_current_neg
                        .iloc[::n, :]
                        .loc[:, ["time_s", "current_plasma_A", "voltage_V"]]
                    )

                    # Subtract 0.115 + i * (1/3) from time_s
                    data_voltage_current_neg["time_s"] = (
                        data_voltage_current_neg["time_s"]
                        - (0.115e-3 + i * (1/f))
                    )

                    # Add the index and loop variable i as new columns
                    data_voltage_current_neg["file_index"] = index + 1

                    data_voltage_current_neg["current_index"] = i + 1

                    # Append the DataFrame to the list
                    file_results_neg.append(data_voltage_current_neg)
                
                # Concatenate the DataFrames in file_results_neg
                file_results_neg_df = pd.concat(
                    file_results_neg, 
                    ignore_index=True
                )
                
                # match the length of info_df to file_results_neg
                repeated_info_df = pd.concat(
                    [info_df]
                    * len(file_results_neg_df),
                    ignore_index=True
                )

                # Concatenate info DataFrame with the data DataFrame
                data_with_info = pd.concat(
                    [repeated_info_df,
                     file_results_neg_df],
                    axis=1
                )

                all_results_neg.append(data_with_info)

            # Combine DataFrames in all_results_neg
            # and save as a .csv file
            neg_csv_path = os.path.join(
                folder_name,
                "current_profiles_neg.csv"
            )

            combined_results = pd.concat(all_results_neg, ignore_index=True)

            combined_results.to_csv(neg_csv_path, index=False)

        except Exception as e:
            print(
                "An error occurred while"
                "extracting the negative current profiles."
                f"Error message: {str(e)}"
            )

        # Positive discharges
        try:
            all_results_pos = []

            for index, file in enumerate(data_frames):
                # Get information from the file path of the CSV file
                info = parse_file_path(psdata_files[index])

                # Convert the information dictionary to a DataFrame
                info_df = pd.DataFrame([info])

                file_results_pos = []
                
                for i in range(0,5):
                    data_voltage_current_pos = file[
                        (file["time_s"] >= 0.280e-3 + i * (1/f))
                        & (file["time_s"] < 0.280e-3 + (i+0.5) * (1/f))
                    ]

                    data_voltage_current_pos = (
                        data_voltage_current_pos
                        .iloc[::n, :]
                        .loc[:, ["time_s", "current_plasma_A", "voltage_V"]]
                    )

                    # Subtract 0.115 + i * (1/3) from time_ms
                    data_voltage_current_pos["time_s"] = (
                        data_voltage_current_pos["time_s"]
                        - (0.280 + i * (1/f))
                    )
                    
                    # Add the index and loop variable i as new columns
                    data_voltage_current_pos["file_index"] = index + 1
                    data_voltage_current_pos["current_index"] = i + 1

                    # Append the DataFrame to the list
                    file_results_pos.append(data_voltage_current_pos)
                
                # Concatenate the DataFrames in file_results_pos
                file_results_pos_df = pd.concat(
                    file_results_pos, 
                    ignore_index=True
                )

                # match the length of info_df to file_results_neg
                repeated_info_df = pd.concat(
                    [info_df]
                    * len(file_results_pos_df),
                    ignore_index=True
                )

                # Concatenate info DataFrame with the data DataFrame
                data_with_info = pd.concat(
                    [repeated_info_df,
                     file_results_pos_df],
                    axis=1
                )

                all_results_pos.append(data_with_info)

            # Combine DataFrames in all_results_neg 
            # and save as a .csv file
            pos_csv_path = os.path.join(
                folder_name,
                "current_profiles_pos.csv"
            )

            combined_results = pd.concat(all_results_pos, ignore_index=True)

            combined_results.to_csv(pos_csv_path, index=False)
        
        except Exception as e:
            print("An error occurred while"
                  "extracting the positive current profiles."
                  f"Error message: {str(e)}")



        # Delete the .csv files after analysis to free up disk space
        for csv_file in csv_files:
            os.remove(csv_file)

In [ ]:
"""column_names_old = [
	'project_name',
	'psdata_file_name',
	'measurement_date',
	'measurement_number',
	'material_supplier',
	'material_name',
	'reaction_type',
	'wattage_const',
	'residence_time_s',
	'plasma_state',
	'power_plasma_W',
	'power_source_W',
	'U_pp_V',
	'current_rms_reactor_A',
	'current_rms_source_A',
	'U_burning_neg_V',
	'U_burning_pos_V',
	'U_burning_avg_V',
	'U_breakdown_neg_V',
	'U_breakdown_pos_V',
	'U_breakdown_avg_V',
	'Q_delta_dis_pos_C',
	'Q_delta_dis_neg_C',
	'Q_delta_dis_avg_C',
	'avg_num_udisch_per_cycle',
	'C_cell_neg_F',
	'C_cell_pos_F',
	'C_cell_avg_F',
	'C_diel_eff_neg_F',
	'C_diel_eff_pos_F',
	'C_diel_eff_avg_F',
	'alpha_neg',
	'alpha_pos',
	'alpha_avg',
	'beta_neg',
	'beta_pos',
	'beta_avg'
]"""

In [ ]:
"""def calculate_results_old(
	data_df,
	f,
	discharge_treshold,
	t, Q, U, Ip, Ib,
	C_diel, C_gas,
	column_names,
	folder_name
):
	
	# Extract some variables from the data
	project_name = data_df['project_name'].iloc[0]
	reaction_type = data_df['reaction_type'].iloc[0]
	material_supplier = data_df['material_supplier'].iloc[0]
	material_name = data_df['material_name'].iloc[0]
	wattage_const = data_df['wattage_const'].iloc[0]
	residence_time_s = data_df['residence_time_s'].iloc[0]
	measurement_number = data_df['measurement_number'].iloc[0]
	plasma_state = data_df['plasma_state'].iloc[0]
	date = data_df['date'].iloc[0]
	
	# Initialize the result DataFrame
	data_to_append_lst = []
	
	# Loop over the grouped data by the column 'psdata_file_name'
	for psdata_file_name, data in data_df.groupby('psdata_file_name'):
			
			# Convert data to numpy array
			data = data.to_numpy()
			
			# Smoothing voltage and current curve
			smoothvoltage = savgol_filter(data[:, U], 5001, 3)
			smoothcurrent1 = savgol_filter(data[:, Ip], 1001, 3)

			# Look in column charge (Q) for all the sign changes
			# and save as 'zero_Q' (Q=0)
			smoothcharge = savgol_filter(data[:, Q], 5001, 3)
			zero_Q = np.where(np.diff(np.sign(smoothcharge)) != 0)

			# Then look in column voltage (U) for the data that
			# are in the same row (Umin)
			zero_dataUmin = data[zero_Q, U]

			# Find the rows that contain pos and neg Umin values
			# in the whole Umin data set.
			posUminrows = np.where(zero_dataUmin > 0)
			negUminrows = np.where(zero_dataUmin < 0)

			# Make arrays containing the Umin data points
			# from the corresponding row numbers.
			posUmindata = zero_dataUmin[posUminrows]
			negUmindata = zero_dataUmin[negUminrows]

			# Calculate the average for both pos and neg Umin.
			posUmin = np.mean(posUmindata)
			negUmin = np.mean(negUmindata)

			# Calculating slopes
			#    B---------------A
			#   /               /
			#  /               /
			# C---------------D

			# Calculating one period (correction to s)
			Period = 1 / f

			# Calculating row of first period start
			t0 = np.where(np.diff(np.sign(data[:, t])) > 0)[0][0]

			# Calculating row of period endings
			t1 = np.argmin(np.abs(data[:, t] - Period * 1))
			t2 = np.argmin(np.abs(data[:, t] - Period * 2))
			t3 = np.argmin(np.abs(data[:, t] - Period * 3))
			t4 = np.argmin(np.abs(data[:, t] - Period * 4))

			# Calculation rows per period
			Periodrows = int(
				np.floor(
					np.mean(
						[t1, t2, t3, t4]
						- np.array([t0, t1, t2, t3])
					)
				)
			)

			# Calculating voltage values and row indexes of
			# points A and C for the first period
			U_A = np.max(data[t0:t1, U])

			U_C = np.min(data[t0:t1, U])

			AU = t0 + int(
				np.floor(
					np.mean(
						np.where(
							data[t0:t1, U] == U_A
						)
					)
				)
			)

			CU = t0 + int(
				np.floor(
					np.mean(
						np.where(
							data[t0:t1, U] == U_C
						)
					)
				)
			)

			U_A1 = np.max(data[t1:t2, U])

			AU1 = t1 + int(
				np.floor(
					np.mean(
						np.where(
							data[t1:t2, U] == U_A1
						)
					)
				)
			)

			# Calculating charge values and row indexes of
			# points A and C for the first period
			Q_A = np.max(data[t0:t1, Q])

			Q_C = np.min(data[t0:t1, Q])

			AQ = t0 + int(
				np.floor(
					np.mean(
						np.where(
							data[t0:t1, Q] == Q_A
						)
					)
				)
			)

			CQ = t0 + int(
				np.floor(
					np.mean(
						np.where(
							data[t0:t1, Q] == Q_C
						)
					)
				)
			)
			
			# Calculate tops of Lissajous
			dx = int(np.floor(1 / 16 * Periodrows))
			step = 100
			
			linfit_AB = []
			linfit_BC = []
			
			for jj in range(AQ + dx, CU - dx, step):

				R2_AB = np.corrcoef(
					smoothvoltage[AQ:jj],
					smoothcharge[AQ:jj]
				)[0, 1] ** 2

				R2_BC = np.corrcoef(
					smoothvoltage[jj:CU],
					smoothcharge[jj:CU]
				)[0, 1] ** 2

				linfit_AB.append(R2_AB)
				
				linfit_BC.append(R2_BC)

			linfit = []

			for kk in range(len(linfit_AB)):

				linfit.append(
					(1 - linfit_AB[kk])
					+ (1 - linfit_BC[kk])
				)

			ind = np.argmin(linfit)
			B = ind * step + AQ + dx

			B1 = B + 1 * Periodrows
			B2 = B + 2 * Periodrows
			B3 = B + 3 * Periodrows

			linfit_CD = []
			linfit_DA = []

			for jj in range(CQ + dx, AU1 - dx, step):

				R2_CD = np.corrcoef(
					smoothvoltage[CQ:jj],
					smoothcharge[CQ:jj]
				)[0, 1] ** 2

				R2_DA = np.corrcoef(
					smoothvoltage[jj:AU1],
					smoothcharge[jj:AU1]
				)[0, 1] ** 2

				linfit_CD.append(R2_CD)

				linfit_DA.append(R2_DA)

			linfit = []

			for kk in range(len(linfit_CD)):

				linfit.append(
					(1 - linfit_CD[kk])
					+ (1 - linfit_DA[kk])
				)

			ind = np.argmin(linfit)

			D = ind * step + CQ + dx

			D1 = D + 1 * Periodrows
			D2 = D + 2 * Periodrows
			D3 = D + 3 * Periodrows

			AQ1 = AQ + 1 * Periodrows
			AQ2 = AQ + 2 * Periodrows
			AQ3 = AQ + 3 * Periodrows
			AQ4 = AQ + 4 * Periodrows
			AU4 = AU + 4 * Periodrows
			AU1 = AU + 1 * Periodrows
			AU2 = AU + 2 * Periodrows
			AU3 = AU + 3 * Periodrows
			AU4 = AU + 4 * Periodrows

			CQ1 = CQ + 1 * Periodrows
			CQ2 = CQ + 2 * Periodrows
			CQ3 = CQ + 3 * Periodrows
			CU1 = CU + 1 * Periodrows
			CU2 = CU + 2 * Periodrows
			CU3 = CU + 3 * Periodrows

			# Finding the values of the first order regression
			# for the data points
			AB = np.polyfit(smoothvoltage[AQ:B], smoothcharge[AQ:B], 1)
			BC = np.polyfit(smoothvoltage[B:CU], smoothcharge[B:CU], 1)
			CD = np.polyfit(smoothvoltage[CQ:D], smoothcharge[CQ:D], 1)
			DA1 = np.polyfit(smoothvoltage[D:AU1], smoothcharge[D:AU1], 1)
			A1B1 = np.polyfit(smoothvoltage[AQ1:B1], smoothcharge[AQ1:B1], 1)
			B1C1 = np.polyfit(smoothvoltage[B1:CU1], smoothcharge[B1:CU1], 1)
			C1D1 = np.polyfit(smoothvoltage[CQ1:D1], smoothcharge[CQ1:D1], 1)
			D1A2 = np.polyfit(smoothvoltage[D1:AU2], smoothcharge[D1:AU2], 1)
			A2B2 = np.polyfit(smoothvoltage[AQ2:B2], smoothcharge[AQ2:B2], 1)
			B2C2 = np.polyfit(smoothvoltage[B2:CU2], smoothcharge[B2:CU2], 1)
			C2D2 = np.polyfit(smoothvoltage[CQ2:D2], smoothcharge[CQ2:D2], 1)
			D2A3 = np.polyfit(smoothvoltage[D2:AU3], smoothcharge[D2:AU3], 1)
			A3B3 = np.polyfit(smoothvoltage[AQ3:B3], smoothcharge[AQ3:B3], 1)
			B3C3 = np.polyfit(smoothvoltage[B3:CU3], smoothcharge[B3:CU3], 1)
			C3D3 = np.polyfit(smoothvoltage[CQ3:D3], smoothcharge[CQ3:D3], 1)
			D3A4 = np.polyfit(smoothvoltage[D3:AU4], smoothcharge[D3:AU4], 1)

			# Calculating the average of the 4 periods.
			meanAB = np.mean([AB, A1B1, A2B2, A3B3], axis=0)
			meanBC = np.mean([BC, B1C1, B2C2, B3C3], axis=0)
			meanCD = np.mean([CD, C1D1, C2D2, C3D3], axis=0)
			meanDA = np.mean([DA1, D1A2, D2A3, D3A4], axis=0)

			# Making arrays to plot the fitted lines.
			xline = np.arange(-10e3, 10.1e3, 1e2)
			lineAB = xline * meanAB[0] + meanAB[1]
			lineBC = xline * meanBC[0] + meanBC[1]
			lineCD = xline * meanCD[0] + meanCD[1]
			lineDA = xline * meanDA[0] + meanDA[1]

			# Plotting the data and fitted lines
			plt.plot(data[t0:t4, U], data[t0:t4, Q], label="Data")
			plt.plot(xline, lineAB, label="AB, C_cell_neg")
			plt.plot(xline, lineBC, label="BC, C_diel_eff_neg")
			plt.plot(xline, lineCD, label="CD, C_cell_pos")
			plt.plot(xline, lineDA, label="DA, C_diel_eff_pos")
			plt.axis([-15e3, 15e3, -1e-6, 1e-6])
			plt.xlabel("Voltage (V)")
			plt.ylabel("Charge (µC)")
			plt.legend()
			
			# Generate a plot filename
			plot_filename = (
				folder_name +
				"/" +
				os.path.splitext(psdata_file_name)[0] +
				"_generated_plot.png"
			)

			# Save the plot in the plot_filename location
			plt.savefig(plot_filename, dpi=150) 
			
			# clear the current plot to avoid plotting data
			# of the previous iteration
			plt.clf()

			# Detect discharges
			smoothcurrent1 = savgol_filter(data[:, Ip], 10001, 3)
			correctedIp = data[:, Ip] - smoothcurrent1
			smoothcurrent2 = savgol_filter(correctedIp, 31, 3)
			smoothcurrent2[smoothcurrent2 < discharge_treshold] = 0

			# Calculating first derivative of current
			# to detect pulses in DA region
			eafgI1 = np.zeros(30)
			for jj in range(D, AQ1):
				eafgI1 = np.append(
					eafgI1,
					(np.mean(smoothcurrent2[jj:(jj+30)])
						- np.mean(smoothcurrent2[(jj-30):jj]))
						/ (30 * (data[1, t] - data[0, t]))
				)

			eafgI2 = np.zeros(30)
			for jj in range(D1, AQ2):
				eafgI2 = np.append(
					eafgI2,
					(np.mean(smoothcurrent2[jj:(jj+30)])
						- np.mean(smoothcurrent2[(jj-30):jj]))
						/ (30 * (data[1, t] - data[0, t]))
			)

			eafgI3 = np.zeros(30)
			for jj in range(D2, AQ3):
				eafgI3 = np.append(
					eafgI3,
					(np.mean(smoothcurrent2[jj:(jj+30)])
						- np.mean(smoothcurrent2[(jj-30):jj]))
						/ (30 * (data[1, t] - data[0, t]))
				)

			eafgI4 = np.zeros(30)
			for jj in range(D3, AQ4):
				eafgI4 = np.append(
					eafgI4,
					(np.mean(smoothcurrent2[jj:(jj+30)])
						- np.mean(smoothcurrent2[(jj-30):jj]))
						/ (30 * (data[1, t] - data[0, t]))
				)

			# Find actual sign changes
			Num_zerosI1 = np.diff(np.sign(eafgI1))
			indx_downI1 = np.where(Num_zerosI1 < 0)[0]
			Num_zerosI2 = np.diff(np.sign(eafgI2))
			indx_downI2 = np.where(Num_zerosI2 < 0)[0]
			Num_zerosI3 = np.diff(np.sign(eafgI3))
			indx_downI3 = np.where(Num_zerosI3 < 0)[0]
			Num_zerosI4 = np.diff(np.sign(eafgI4))
			indx_downI4 = np.where(Num_zerosI4 < 0)[0]

			# Counting times that first derivative
			# has zero points from pos to neg
			Num_peaksI = (
				len(indx_downI1)
				+ len(indx_downI2)
				+ len(indx_downI3)
				+ len(indx_downI4)
			)
			
			Av_num_peaksI = round(Num_peaksI / 4)

			# Displaced charge per cycle (C)
			Q_null = meanAB[1] - meanCD[1]

			# Calculate plasma power integrated over four periods (W)
			plasmapower = lissajous_area(
				data[t0:t4, U], data[t0:t4, Q]
			) * 1e3
			
			# Calculating power source power (W)
			sourcepower = np.mean(data[t0:t4, U] * data[t0:t4, Ib])

			# Calculating RMS current plasma (A)
			RMSIp = np.sqrt(np.mean(np.square(data[t0:t4, Ip])))

			# Calculating RMS current source (A)
			RMSIb = np.sqrt(np.mean(np.square(data[t0:t4, Ib])))

			# Calculating Upp (V)
			Upp = abs(U_A) + abs(U_C)
			
			# Variables for the burning and breakdown voltage
			U_delta_pos = posUmin
			U_delta_neg = negUmin
			
			C_cell_pos = meanCD[0]
			C_cell_neg = meanAB[0]
			C_cell_avg = (C_cell_pos + C_cell_neg) / 2
			
			C_diel_eff_pos = meanDA[0]
			C_diel_eff_neg = meanBC[0]
			C_diel_eff_avg = (C_diel_eff_pos + C_diel_eff_neg) / 2
			
			alpha_pos = (C_diel - C_diel_eff_pos) / (C_diel - C_cell_pos)
			alpha_neg = (C_diel - C_diel_eff_neg) / (C_diel - C_cell_neg)
			alpha_avg = (alpha_pos + alpha_neg) / 2
			
			beta_pos = (C_diel_eff_pos - C_cell_pos) / (C_diel - C_cell_pos)
			beta_neg = (C_diel_eff_neg - C_cell_neg) / (C_diel - C_cell_neg)
			beta_avg = (beta_pos + beta_neg) / 2

			# Calculate burning voltage (V)
			U_burning_pos = (
				(1 - C_cell_pos / C_diel)
				/ (1 - C_cell_pos / C_diel_eff_pos)
				* (U_delta_pos)
			)	
			U_burning_neg = (
				(1 - C_cell_neg / C_diel)
				/ (1 - C_cell_neg / C_diel_eff_neg)
				* (U_delta_neg)
			)
			U_burning_avg = (np.abs(U_burning_neg) + U_burning_pos) / 2
			
			# Calculate breakdown voltage
			U_breakdown_neg = 1 / (1 + C_gas/C_diel) * U_delta_neg
			U_breakdown_pos = 1 / (1 + C_gas/C_diel) * U_delta_pos
			U_breakdown_avg = (np.abs(U_breakdown_neg) + U_breakdown_pos) / 2
			
			# Conductively transfered charge
			Q_delta_dis_pos = Q_null / (1 - C_cell_pos / C_diel)
			Q_delta_dis_neg = Q_null / (1 - C_cell_neg / C_diel)
			Q_delta_dis_avg = (Q_delta_dis_pos + Q_delta_dis_neg) / 2
			
			# Write the results to the result DataFrame
			data_to_append_lst.append(
				{
					'project_name': project_name,
					'psdata_file_name': psdata_file_name,
					'measurement_date': date,
					'measurement_number': measurement_number,
					'material_supplier': material_supplier,
					'material_name': material_name,
					'reaction_type': reaction_type,
					'wattage_const': wattage_const,
					'residence_time_s': residence_time_s,
					'plasma_state': plasma_state,
					'power_plasma_W': plasmapower,
					'power_source_W': sourcepower,
					'U_pp_V': Upp,
					'current_rms_reactor_A': RMSIp,
					'current_rms_source_A': RMSIb,
					'U_burning_neg_V': U_burning_neg,
					'U_burning_pos_V': U_burning_pos,
					'U_burning_avg_V': U_burning_avg,
					'U_breakdown_neg_V': U_breakdown_neg,
					'U_breakdown_pos_V': U_breakdown_pos,
					'U_breakdown_avg_V': U_breakdown_avg,
					'Q_delta_dis_pos_C': Q_delta_dis_pos,
					'Q_delta_dis_neg_C': Q_delta_dis_neg,
					'Q_delta_dis_avg_C': Q_delta_dis_avg,
					'avg_num_udisch_per_cycle': Av_num_peaksI,
					'C_cell_neg_F': C_cell_neg,
					'C_cell_pos_F': C_cell_pos,
					'C_cell_avg_F': C_cell_avg,
					'C_diel_eff_neg_F': C_diel_eff_neg,
					'C_diel_eff_pos_F': C_diel_eff_pos,
					'C_diel_eff_avg_F': C_diel_eff_avg,
					'alpha_neg': alpha_neg,
					'alpha_pos': alpha_pos,
					'alpha_avg': alpha_avg,
					'beta_neg': beta_neg,
					'beta_pos': beta_pos,
					'beta_avg': beta_avg
				}
			)
	
	# Create a DataFrame from the data_to_append_lst
	result_df = pd.DataFrame(
		data_to_append_lst,
		columns=column_names
	)
	
	# Return the result_df & figures dict
	return result_df"""